# Manufacturing Transition Stability Analysis

This notebook is a **public technical reconstruction** of the analytical approach used in a Duke University MQM capstone.

**Important:** All data in this notebook is synthetic. The original client work was completed in a secure environment. No client data, original code, project-specific findings, or confidential deliverables are reproduced here.


## 1. Business Problem

A manufacturing process can hit a new target value quickly while still exhibiting high variability. This means **target achievement** and **true process stability** are not necessarily the same thing.

This notebook demonstrates how to:
- measure transition performance,
- compare stabilization behavior across transition types,
- investigate potential drivers,
- and prioritize operational review using frequency and impact.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("data/synthetic_transition_data.csv")
df.head()


## 2. Data Overview

The synthetic dataset contains transition-level manufacturing observations such as:
- previous and new product grade,
- operating shift,
- production speed,
- time to reach the target,
- process variability,
- stabilization time,
- and a synthetic operational-impact score.


In [ ]:
print(f"Rows: {len(df):,}")
print(f"Transition types: {df['transition_type'].nunique()}")
print("\nMissing values:")
display(df.isna().sum().to_frame("missing"))


## 3. Target Achievement vs. Process Stability

A process may be close to the target but still be too variable to consider stable. The synthetic dataset includes:
- `target_gap_pct`: distance from the target,
- `process_variability`: a synthetic variability measure,
- `stable_flag`: a synthetic demonstration rule combining target proximity and variability.


In [ ]:
stability_summary = (
    df.groupby("stable_flag")
      .agg(
          observations=("transition_id", "count"),
          avg_target_gap_pct=("target_gap_pct", "mean"),
          avg_variability=("process_variability", "mean"),
          avg_stabilization_time=("stabilization_time_min", "mean"),
      )
)
stability_summary.index = stability_summary.index.map({0: "Not Stable", 1: "Stable"})
stability_summary.round(2)


## 4. Transition-Level Comparison


In [ ]:
transition_summary = (
    df.groupby("transition_type", as_index=False)
      .agg(
          transition_frequency=("transition_id", "count"),
          avg_stabilization_time=("stabilization_time_min", "mean"),
          avg_variability=("process_variability", "mean"),
          avg_impact=("operational_impact_score", "mean"),
      )
      .sort_values("avg_stabilization_time", ascending=False)
)
transition_summary.round(2)


In [ ]:
plot_df = transition_summary.sort_values("avg_stabilization_time")

plt.figure(figsize=(9, 5.2))
plt.barh(plot_df["transition_type"], plot_df["avg_stabilization_time"])
plt.xlabel("Average stabilization time (minutes)")
plt.ylabel("Transition type")
plt.title("Synthetic Stabilization Time by Transition Type")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Operational Prioritization

A transition that is highly impactful but rare may require a different response than one that is both frequent and high-impact.

The matrix below uses the **median synthetic frequency and impact** as demonstration cutoffs. These thresholds are illustrative, not client rules.


In [ ]:
x = transition_summary["transition_frequency"]
y = transition_summary["avg_impact"]

x_mid = x.median()
y_mid = y.median()

plt.figure(figsize=(8.6, 5.6))
plt.scatter(x, y, s=80)

for _, row in transition_summary.iterrows():
    short_label = row["transition_type"].replace("Grade_", "")
    plt.annotate(
        short_label,
        (row["transition_frequency"], row["avg_impact"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
    )

plt.axvline(x_mid, linestyle="--", linewidth=1)
plt.axhline(y_mid, linestyle="--", linewidth=1)
plt.xlabel("Transition frequency")
plt.ylabel("Average operational impact score")
plt.title("Synthetic Operational Prioritization Matrix")
plt.grid(alpha=0.20)
plt.tight_layout()
plt.show()


## 6. Predictive Driver Analysis

To demonstrate how potential transition drivers can be investigated, a Random Forest model is trained to predict synthetic stabilization time.

Because the dataset itself is artificial, **feature importance values are demonstration outputs only** and should not be interpreted as findings from the original capstone.


In [ ]:
features = [
    "transition_type",
    "shift",
    "production_speed",
    "time_to_target_min",
    "process_variability",
    "target_gap_pct",
]

X = df[features]
y = df["stabilization_time_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

categorical = ["transition_type", "shift"]
numeric = ["production_speed", "time_to_target_min", "process_variability", "target_gap_pct"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", "passthrough", numeric),
])

model = RandomForestRegressor(
    n_estimators=250,
    random_state=42,
    min_samples_leaf=3
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, pred):.2f} minutes")
print(f"R²:  {r2_score(y_test, pred):.3f}")


In [ ]:
feature_names = list(
    pipeline.named_steps["preprocessor"]
    .named_transformers_["categorical"]
    .get_feature_names_out(categorical)
) + numeric

importance = pipeline.named_steps["model"].feature_importances_

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance,
}).sort_values("importance", ascending=False)

feature_importance.head(10)


In [ ]:
top = feature_importance.head(10).copy()
top["feature"] = (
    top["feature"]
    .str.replace("transition_type_", "Transition: ", regex=False)
    .str.replace("shift_", "Shift: ", regex=False)
    .str.replace("_", " ", regex=False)
)

top = top.sort_values("importance")

plt.figure(figsize=(9, 5.4))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Relative feature importance")
plt.title("Synthetic Drivers of Stabilization Time")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Business Interpretation

This synthetic demonstration shows how an operations team could use transition data to:

1. **Separate speed from stability** — hitting a target is not enough if the process remains variable.
2. **Compare transition types** — use common measures of stabilization time and variability.
3. **Investigate potential drivers** — evaluate settings, timing, shift, and transition characteristics.
4. **Prioritize action** — combine frequency and impact rather than treating every transition equally.

The exact synthetic rankings and model outputs in this notebook are **not findings from the original capstone**.


## Confidentiality Note

The original capstone was conducted in a secure environment. This notebook does not contain or reconstruct proprietary client data, original client code, confidential project details, or actual client findings.

All data and results shown here were generated specifically for this public portfolio demonstration.
